In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/03 16:40:39 WARN Utils: Your hostname, abdo-HP-ZBook-17-G3, resolves to a loopback address: 127.0.1.1; using 192.168.1.6 instead (on interface wlp2s0)
26/03/03 16:40:39 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/03 16:40:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
total_cores = spark.sparkContext.defaultParallelism

In [3]:
total_cores

4

In [4]:
import pandas as pd
import time


data = "EuropePrices12-13-25.csv"

pandas_df = pd.read_csv(data)
spark_df = spark.read.csv(data,header=True,inferSchema=True)

In [5]:
spark_df.printSchema()

root
 |-- price: integer (nullable = true)
 |-- timestamp: timestamp (nullable = true)



In [ ]:
spark_df.show()

+-----+-------------------+
|price|          timestamp|
+-----+-------------------+
|   45|2024-04-24 13:00:00|
|  145|2024-04-24 14:00:00|
|  228|2024-04-24 15:00:00|
|  395|2024-04-24 16:00:00|
|  863|2024-04-24 17:00:00|
| 1077|2024-04-24 18:00:00|
| 1009|2024-04-24 19:00:00|
| 1027|2024-04-24 20:00:00|
| 1137|2024-04-24 21:00:00|
| 1343|2024-04-24 22:00:00|
| 1461|2024-04-24 23:00:00|
| 1718|2024-04-25 00:00:00|
| 1897|2024-04-25 01:00:00|
| 2249|2024-04-25 02:00:00|
| 2275|2024-04-25 03:00:00|
| 2238|2024-04-25 04:00:00|
| 2172|2024-04-25 05:00:00|
| 2129|2024-04-25 06:00:00|
| 2107|2024-04-25 07:00:00|
| 2004|2024-04-25 08:00:00|
+-----+-------------------+
only showing top 20 rows


In [7]:
spark_df.count()

15234

In [8]:
pandas_df.count()

price        15234
timestamp    15234
dtype: int64

In [9]:
spark_df.describe().show()

+-------+------------------+
|summary|             price|
+-------+------------------+
|  count|             15234|
|   mean| 7054.934423001181|
| stddev|12200.706478220101|
|    min|                 1|
|    max|            400000|
+-------+------------------+



In [10]:
pandas_df.describe()

,price
count,15234.000000
mean,7054.934423
std,12200.706478
min,1.000000
25%,4918.000000
50%,6634.000000
75%,7049.000000
max,400000.000000


In [11]:
start_spark = time.perf_counter()

price_between_3000_4000 = spark_df.where(spark_df.price.between(3000,4000))

end_spark = time.perf_counter()
print(f"Spark Time: {end_spark - start_spark:.4f} seconds")

Spark Time: 0.0694 seconds


In [12]:
price_between_3000_4000.describe().show()

+-------+-----------------+
|summary|            price|
+-------+-----------------+
|  count|              813|
|   mean|3622.212792127921|
| stddev|220.6624436177393|
|    min|             3000|
|    max|             4000|
+-------+-----------------+



In [13]:
start_pandas = time.perf_counter()

price_between_3000_4000 = pandas_df.where(pandas_df.price.between(3000,4000))

end_pandas = time.perf_counter()
print(f"Pandas Time: {end_pandas - start_pandas:.4f} seconds")

Pandas Time: 0.0185 seconds


In [14]:
price_between_3000_4000.describe()

,price
count,813.000000
mean,3622.212792
std,220.662444
min,3000.000000
25%,3497.000000
50%,3627.000000
75%,3797.000000
max,4000.000000


In [15]:
start_spark = time.perf_counter()

price_between_3000_4000 = spark_df.where(spark_df.price.between(3000,4000))
price_between_3000_4000.describe().show()

end_spark = time.perf_counter()
print(f"Spark Time: {end_spark - start_spark:.4f} seconds")

+-------+-----------------+
|summary|            price|
+-------+-----------------+
|  count|              813|
|   mean|3622.212792127921|
| stddev|220.6624436177393|
|    min|             3000|
|    max|             4000|
+-------+-----------------+

Spark Time: 0.3484 seconds


In [16]:
start_pandas = time.perf_counter()

price_between_3000_4000 = pandas_df.where(pandas_df.price.between(3000,4000))
print(price_between_3000_4000.describe())

end_pandas = time.perf_counter()
print(f"Pandas Time: {end_pandas - start_pandas:.4f} seconds")

             price
count   813.000000
mean   3622.212792
std     220.662444
min    3000.000000
25%    3497.000000
50%    3627.000000
75%    3797.000000
max    4000.000000
Pandas Time: 0.0283 seconds


In [17]:
%%timeit -n 5 -r 2

spark_df.where(spark_df.price.between(3000, 4000)).count()

309 ms ± 26.1 ms per loop (mean ± std. dev. of 2 runs, 5 loops each)


In [18]:
%%timeit -n 5 -r 2

pandas_df.where(pandas_df.price.between(3000, 4000)).count()

6.31 ms ± 1.48 ms per loop (mean ± std. dev. of 2 runs, 5 loops each)


In [19]:
from pyspark.sql import functions as sf
spark_df_month = spark_df.select("price", sf.month('timestamp'),sf.year('timestamp'))


In [20]:
spark_df_month.show(n=5)

+-----+----------------+---------------+
|price|month(timestamp)|year(timestamp)|
+-----+----------------+---------------+
|   45|               4|           2024|
|  145|               4|           2024|
|  228|               4|           2024|
|  395|               4|           2024|
|  863|               4|           2024|
+-----+----------------+---------------+
only showing top 5 rows


In [21]:
start_spark = time.perf_counter()
spark_df.groupBy(sf.date_format("timestamp", "yyyy-MM").alias("month_year")) \
    .agg(
        sf.count("price").alias("record_count"),
        sf.sum("price").alias("total_price"),
        sf.min("price").alias("min_price"),
        sf.max("price").alias("max_price"),
        sf.mean("price").alias("avg_price")
    ) \
    .orderBy("month_year") \
    .show(n=21)
end_spark = time.perf_counter()
print(f"Spark Time: {end_spark - start_spark:.4f} seconds")

+----------+------------+-----------+---------+---------+------------------+
|month_year|record_count|total_price|min_price|max_price|         avg_price|
+----------+------------+-----------+---------+---------+------------------+
|   2024-04|         166|     480624|       45|     4100|2895.3253012048194|
|   2024-05|         765|    2842572|      625|    10000| 3715.780392156863|
|   2024-06|         728|    2987185|     2600|     4366|4103.2760989010985|
|   2024-07|         775|    3913489|     4151|   100100| 5049.663225806452|
|   2024-08|         747|    4039446|     4567|   269683| 5407.558232931727|
|   2024-09|         729|    5533497|     4781|   400000| 7590.530864197531|
|   2024-10|         759|    4702217|     1013|     6925| 6195.279314888011|
|   2024-11|         826|    5349289|        4|    68000| 6476.136803874092|
|   2024-12|         908|    5838578|        1|     8808| 6430.151982378855|
|   2025-01|         796|    5816694|        1|    36330| 7307.404522613066|

In [22]:
start_pandas = time.perf_counter()

pandas_df['month_year'] = pd.to_datetime(pandas_df['timestamp']).dt.to_period('M').astype(str)
print(pandas_df.groupby('month_year')['price'].agg(['count', 'sum', 'min', 'max', 'mean']).head(n=21))

end_pandas = time.perf_counter()
print(f"Pandas Time: {end_pandas - start_pandas:.4f} seconds")

            count       sum   min     max          mean
month_year                                             
2024-04       166    480624    45    4100   2895.325301
2024-05       765   2842572   625   10000   3715.780392
2024-06       728   2987185  2600    4366   4103.276099
2024-07       775   3913489  4151  100100   5049.663226
2024-08       747   4039446  4567  269683   5407.558233
2024-09       729   5533497  4781  400000   7590.530864
2024-10       759   4702217  1013    6925   6195.279315
2024-11       826   5349289     4   68000   6476.136804
2024-12       908   5838578     1    8808   6430.151982
2025-01       796   5816694     1   36330   7307.404523
2025-02       670   4650696  6585    7090   6941.337313
2025-03       746   4936071  6315    6791   6616.717158
2025-04       724   4798578  6530    6710   6627.870166
2025-05       799   5204878  4000    6650   6514.240300
2025-06      1030   9600949    20  222222   9321.309709
2025-07       837  10778295     1  222222  12877

In [23]:
%%timeit -n 5 -r 2

spark_df.groupBy(sf.date_format("timestamp", "yyyy-MM").alias("month_year")) \
    .agg(
        sf.count("price").alias("record_count"),
        sf.sum("price").alias("total_price"),
        sf.min("price").alias("min_price"),
        sf.max("price").alias("max_price"),
        sf.mean("price").alias("avg_price")
    ) \
    .orderBy("month_year") \
    .collect()

496 ms ± 129 ms per loop (mean ± std. dev. of 2 runs, 5 loops each)


In [24]:
%%timeit -n 5 -r 2

pandas_df['month_year'] = pd.to_datetime(pandas_df['timestamp']).dt.to_period('M').astype(str)
pandas_df.groupby('month_year')['price'].agg(['count', 'sum', 'min', 'max', 'mean'])

22.9 ms ± 5.38 ms per loop (mean ± std. dev. of 2 runs, 5 loops each)
